# WandB Plotting for Master Thesis

This notebook demonstrates how to create publication-quality plots from your wandb logs.
Perfect for creating figures for your master thesis on ACT networks.

## Setup

In [ ]:
# Import the plotting system
from data_loader import WandbDataLoader, load_multiple_experiments
from plotting import PlotManager, PlotConfig, PlotConfigs
from export import ExportManager
import matplotlib.pyplot as plt
import pandas as pd

# Enable inline plotting
%matplotlib inline

## Configuration

In [ ]:
# Set your wandb directory path
WANDB_DIR = "../wandb"  # Adjust this path to your wandb directory
OUTPUT_DIR = "notebook_plots"

# Choose configuration
config = PlotConfigs.thesis_config()
plot_manager = PlotManager(config)
export_manager = ExportManager(OUTPUT_DIR)

## Load Data

In [ ]:
# Load data from wandb logs
loader = WandbDataLoader(WANDB_DIR)
runs_data = loader.load_all_runs()

print(f"Found {len(runs_data)} runs")
for run_id, data in runs_data.items():
    print(f"Run {run_id}: {data.shape[0]} data points")
    print(f"  Columns: {list(data.columns)}")
    print()

## Data Preview

In [ ]:
# Show data preview
if runs_data:
    sample_data = list(runs_data.values())[0]
    print("Sample data:")
    display(sample_data.head())
    
    print("\nData statistics:")
    display(sample_data.describe())

## Training Curves

In [ ]:
# Create training curves
metrics = ["train/loss", "val/loss", "train/accuracy", "val/accuracy"]

if len(runs_data) > 1:
    # Multiple runs comparison
    fig = plot_manager.create_training_curves(
        runs_data, metrics,
        title="Training Curves Comparison",
        smooth=True
    )
else:
    # Single run
    fig = plot_manager.create_training_curves(
        list(runs_data.values())[0], metrics,
        title="ACT Training Progress",
        smooth=True
    )

plt.show()

# Export the figure
export_manager.export_figure(fig, "training_curves_notebook", ["png", "pdf", "svg"])

## Loss Comparison

In [ ]:
# Create loss comparison plot
if len(runs_data) > 1:
    fig = plot_manager.create_comparison_plot(
        runs_data, "val/loss",
        title="Validation Loss Comparison",
        final_values=True
    )
    plt.show()
    export_manager.export_figure(fig, "loss_comparison_notebook", ["png", "pdf"])
else:
    print("Need multiple runs for comparison plots")

## Learning Rate Schedule

In [ ]:
# Create learning rate schedule plot
if "train/learning_rate" in list(runs_data.values())[0].columns:
    fig = plot_manager.create_learning_rate_schedule(
        runs_data if len(runs_data) > 1 else list(runs_data.values())[0],
        title="Learning Rate Schedule"
    )
    plt.show()
    export_manager.export_figure(fig, "lr_schedule_notebook", ["png", "pdf"])
else:
    print("No learning rate data found")

## Custom Plot Example

In [ ]:
# Create a custom plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

if runs_data:
    sample_data = list(runs_data.values())[0]
    
    # Plot 1: Loss curves
    ax1.plot(sample_data['step'], sample_data['train/loss'], label='Training Loss', color='blue')
    ax1.plot(sample_data['step'], sample_data['val/loss'], label='Validation Loss', color='red')
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss Curves')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Accuracy curves
    ax2.plot(sample_data['step'], sample_data['train/accuracy'], label='Training Accuracy', color='green')
    ax2.plot(sample_data['step'], sample_data['val/accuracy'], label='Validation Accuracy', color='orange')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy Curves')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Export custom plot
export_manager.export_figure(fig, "custom_plot_notebook", ["png", "pdf"])

## Generate LaTeX Code

In [ ]:
# Generate LaTeX code for thesis inclusion
latex_code = export_manager.generate_latex_figure(
    "training_curves_notebook.pdf",
    "Training curves for the ACT network showing loss and accuracy progression over training steps.",
    "act_training_curves"
)

print("LaTeX code for your thesis:")
print("=" * 50)
print(latex_code)
print("=" * 50)

# Save LaTeX code to file
export_manager.save_latex_code(latex_code, "thesis_figure_code")

## Summary

In [ ]:
# Show summary of created files
import os
from pathlib import Path

output_path = Path(OUTPUT_DIR)
if output_path.exists():
    files = list(output_path.glob("*"))
    print(f"Created {len(files)} files in {OUTPUT_DIR}/:")
    for file in sorted(files):
        size = file.stat().st_size
        print(f"  📄 {file.name} ({size:,} bytes)")
    
    print("\n💡 Tips for thesis writing:")
    print("1. Use PDF files for LaTeX documents (vector graphics)")
    print("2. Use PNG files for presentations")
    print("3. Copy the generated LaTeX code to include figures")
    print("4. SVG files can be edited in vector graphics software")
else:
    print("No output directory found")